In [ ]:
import pandas as pd

names = pd.read_csv('data/country_name.csv', skipinitialspace=True)
areas = pd.read_csv('data/country_area.csv', skipinitialspace=True)
iso   = pd.read_csv('data/country_name_iso_3166.csv', skipinitialspace=True)
pop   = pd.read_csv('data/country_population.csv', skipinitialspace=True)
gdp   = pd.read_csv('data/country_gdp.csv', skipinitialspace=True)

# Normalise join key
names.columns = names.columns.str.strip()
areas.columns = areas.columns.str.strip()
iso.columns   = iso.columns.str.strip()
pop.columns   = pop.columns.str.strip()
gdp.columns   = gdp.columns.str.strip()

print(names.shape, areas.shape, iso.shape, pop.shape, gdp.shape)

In [ ]:
# Merge all tables on Alpha-3 code
df = (
    iso
    .merge(names.rename(columns={'alpha_3_code': 'alpha_3_code'}),
           on='alpha_3_code', how='left')
    .merge(areas.rename(columns={'alpha_3_code': 'alpha_3_code'}),
           on='alpha_3_code', how='left')
    .merge(pop, on='alpha_3_code', how='left')
    .merge(gdp, on='alpha_3_code', how='left')
)

df['total_area'] = df['land'] + df['water'].fillna(0)
df.head()

In [ ]:
# 10 largest countries by land area
df.nlargest(10, 'land')[['english_short_name', 'alpha_3_code', 'land', 'water']]

In [ ]:
# 10 smallest countries by land area (excluding zero/unknown)
df[df['land'] > 0].nsmallest(10, 'land')[['english_short_name', 'alpha_3_code', 'land', 'water']]

In [ ]:
import matplotlib.pyplot as plt

top20 = df.nlargest(20, 'land').sort_values('land')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20['english_short_name'], top20['land'] / 1_000_000)
ax.set_xlabel('Land area (million km²)')
ax.set_title('Top 20 countries by land area')
plt.tight_layout()
plt.show()

## Population (2024)\nSource: World Bank SP.POP.TOTL

In [ ]:
# 10 most populous countries (2024)
df.nlargest(10, 'population')[['english_short_name', 'alpha_3_code', 'population']]

In [ ]:
# 10 least populous countries (2024, excluding zero/null)
df[df['population'] > 0].nsmallest(10, 'population')[['english_short_name', 'alpha_3_code', 'population']]

In [ ]:
import matplotlib.pyplot as plt

top20_pop = df.nlargest(20, 'population').sort_values('population')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_pop['english_short_name'], top20_pop['population'] / 1_000_000)
ax.set_xlabel('Population (millions)')
ax.set_title('Top 20 countries by population (2024)')
plt.tight_layout()
plt.show()

In [ ]:
# Population density (people per km² of land)
df['pop_density'] = df['population'] / df['land']

df[df['land'] > 0].nlargest(10, 'pop_density')[['english_short_name', 'alpha_3_code', 'population', 'land', 'pop_density']].round(1)

In [ ]:
top20_density = df[df['land'] > 0].nlargest(20, 'pop_density').sort_values('pop_density')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_density['english_short_name'], top20_density['pop_density'])
ax.set_xlabel('Population density (people per km²)')
ax.set_title('Top 20 countries by population density (2024)')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.colors as mcolors

plot_df = df[df['land'] > 0].dropna(subset=['population', 'pop_density']).copy()

norm = mcolors.LogNorm(vmin=plot_df['pop_density'].min(), vmax=plot_df['pop_density'].max())

fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(
    plot_df['land'] / 1_000_000,
    plot_df['population'] / 1_000_000,
    c=plot_df['pop_density'],
    cmap='YlOrRd',
    norm=norm,
    s=50,
    alpha=0.85,
    edgecolors='grey',
    linewidths=0.3
)
plt.colorbar(sc, ax=ax, label='Population density (people / km², log scale)')
ax.set_xlabel('Land area (million km²)')
ax.set_ylabel('Population (millions)')
ax.set_title('Land area vs Population (2024) — colour = density (log scale)')

for _, row in plot_df.nlargest(10, 'population').iterrows():
    ax.annotate(row['alpha_3_code'],
                xy=(row['land'] / 1_000_000, row['population'] / 1_000_000),
                fontsize=7, ha='left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

top20_gdp = df.nlargest(20, 'gdp_usd').sort_values('gdp_usd')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_gdp['english_short_name'], top20_gdp['gdp_usd'] / 1_000_000_000_000)
ax.set_xlabel('GDP (trillion USD)')
ax.set_title('Top 20 countries by GDP (2024)')
plt.tight_layout()
plt.show()

In [ ]:
# 10 largest economies by GDP (2024)
df.nlargest(10, 'gdp_usd')[['english_short_name', 'alpha_3_code', 'gdp_usd']]